# 12.14 Knowledge graphs & embeddings (TransE)

TransE models a fact as head plus relation near tail. TransE represents a typed fact as head plus relation near tail. This walkthrough implements a small seeded TransE trainer and ranks knowledge-graph tails with MRR across a graph ladder. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 1215
random.seed(SEED)
np.random.seed(SEED)


def make_d1_graph():
    graph = nx.Graph()
    graph.add_nodes_from(range(4))
    graph.add_edges_from([(0, 1), (0, 2), (1, 2), (2, 3)])
    labels = {0: 0, 1: 0, 2: 1, 3: 1}
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_karate_rung():
    graph = nx.karate_club_graph()
    labels = {}
    for node, data in graph.nodes(data=True):
        labels[node] = 0 if data.get("club") == "Mr. Hi" else 1
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_sbm_rung(sizes, p_in, p_out, seed, noise_edges=0):
    probs = []
    for row in range(len(sizes)):
        prob_row = []
        for col in range(len(sizes)):
            prob_row.append(p_in if row == col else p_out)
        probs.append(prob_row)
    graph = nx.stochastic_block_model(sizes, probs, seed=seed)
    rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    for _ in range(noise_edges):
        u = int(rng.choice(nodes))
        v = int(rng.choice(nodes))
        if u != v:
            graph.add_edge(u, v)
    labels = {}
    offset = 0
    for block, size in enumerate(sizes):
        for node in range(offset, offset + size):
            labels[node] = block
        offset = offset + size
    positions = nx.spring_layout(graph, seed=seed)
    return graph, labels, positions


def make_cora_like_rung():
    graph, labels, positions = make_sbm_rung([18, 16, 14], 0.26, 0.035, SEED + 4, noise_edges=8)
    rng = np.random.default_rng(SEED + 44)
    years = {}
    node_types = {}
    for node in graph.nodes():
        years[node] = int(2016 + rng.integers(0, 7))
        node_types[node] = "paper"
    nx.set_node_attributes(graph, years, "year")
    nx.set_node_attributes(graph, node_types, "kind")
    return graph, labels, positions


def make_large_noisy_rung():
    graph, labels, positions = make_sbm_rung([35, 32, 30, 28], 0.18, 0.045, SEED + 5, noise_edges=55)
    rng = np.random.default_rng(SEED + 55)
    for u, v in graph.edges():
        graph[u][v]["time"] = int(rng.integers(0, 10))
        graph[u][v]["relation"] = "strong" if labels[u] == labels[v] else "weak"
    return graph, labels, positions


def build_graph_ladder():
    d1_graph, d1_labels, d1_pos = make_d1_graph()
    d2_graph, d2_labels, d2_pos = make_karate_rung()
    d3_graph, d3_labels, d3_pos = make_sbm_rung([12, 12, 12], 0.32, 0.04, SEED + 3, noise_edges=5)
    d4_graph, d4_labels, d4_pos = make_cora_like_rung()
    d5_graph, d5_labels, d5_pos = make_large_noisy_rung()
    return [
        {"name": "D1 toy", "graph": d1_graph, "labels": d1_labels, "pos": d1_pos},
        {"name": "D2 karate", "graph": d2_graph, "labels": d2_labels, "pos": d2_pos},
        {"name": "D3 SBM", "graph": d3_graph, "labels": d3_labels, "pos": d3_pos},
        {"name": "D4 Cora-like subset", "graph": d4_graph, "labels": d4_labels, "pos": d4_pos},
        {"name": "D5 larger noisy", "graph": d5_graph, "labels": d5_labels, "pos": d5_pos},
    ]


def partition_from_labels(labels):
    groups = defaultdict(set)
    for node, label in labels.items():
        groups[label].add(node)
    return list(groups.values())


def accuracy_score(y_true, y_pred):
    correct = 0
    total = 0
    for node, truth in y_true.items():
        if node in y_pred:
            correct = correct + int(y_pred[node] == truth)
            total = total + 1
    return correct / max(total, 1)


## The concept, built once (D1): TransE scoring

TransE scores a triple by $f(h,r,t)=\lVert e_h+e_r-e_t\rVert_2$ and trains with $\max(0,\gamma+f(h,r,t)-f(h',r,t'))$. The lesson checks distance $0.000$, corrupt distance $\sqrt2=1.414$, zero margin loss, inverse relation $[-2,0]$, and true tail ranked first.

In [ ]:

def transe_score(entity_embeddings, relation_embeddings, triple):
    head, relation, tail = triple
    residual = entity_embeddings[head] + relation_embeddings[relation] - entity_embeddings[tail]
    return float(np.linalg.norm(residual, ord=2))


def margin_loss(pos_score, neg_score, margin=1.0):
    return max(0.0, margin + pos_score - neg_score)


def transe_rank(entity_embeddings, relation_embeddings, head, relation, candidates):
    scores = []
    for tail in candidates:
        score = transe_score(entity_embeddings, relation_embeddings, (head, relation, tail))
        scores.append((tail, score))
    scores.sort(key=lambda item: item[1])
    return scores


## Rank candidate tails

The same score orders true and corrupted facts; lower distance means a better candidate tail.

In [ ]:

entity_embeddings = {
    "h": np.array([1.0, 1.0]),
    "t": np.array([3.0, 1.0]),
    "corrupt": np.array([2.0, 2.0]),
}
relation_embeddings = {
    "r": np.array([2.0, 0.0]),
    "r_inverse": np.array([-2.0, 0.0]),
}
positive_distance = transe_score(entity_embeddings, relation_embeddings, ("h", "r", "t"))
negative_distance = transe_score(entity_embeddings, relation_embeddings, ("h", "r", "corrupt"))
loss = margin_loss(positive_distance, negative_distance, margin=1.0)
ranking = transe_rank(entity_embeddings, relation_embeddings, "h", "r", ["corrupt", "t"])
print("positive distance", round(positive_distance, 3))
print("corrupt distance", round(negative_distance, 3))
print("margin loss", round(loss, 3))
print("inverse relation", relation_embeddings["r_inverse"].tolist())
print("ranking", ranking)
assert round(positive_distance, 3) == 0.000
assert round(negative_distance, 3) == 1.414
assert round(loss, 3) == 0.000
assert relation_embeddings["r_inverse"].tolist() == [-2.0, 0.0]
assert ranking[0][0] == "t"


## The dataset ladder

D1 is a tiny KG; D2-D5 convert graph edges into typed same-community and cross-community facts.

In [ ]:

ladder = build_graph_ladder()
for rung in ladder:
    graph = rung["graph"]
    labels = rung["labels"]
    sample_nodes = list(graph.nodes())[:5]
    sample_edges = list(graph.edges())[:5]
    print(rung["name"])
    print("  nodes", graph.number_of_nodes(), "edges", graph.number_of_edges(), "classes", sorted(set(labels.values())))
    print("  sample nodes", sample_nodes)
    print("  sample edges", sample_edges)


## Run the same method across D1-D5

In [ ]:

def graph_to_kg(rung):
    graph = rung["graph"]
    labels = rung["labels"]
    triples = []
    for u, v in graph.edges():
        relation = "same" if labels[u] == labels[v] else "cross"
        triples.append((u, relation, v))
        triples.append((v, relation, u))
    return triples


def train_transe(triples, nodes, relations, dim=8, epochs=80, lr=0.03, margin=1.0, seed=SEED):
    rng = np.random.default_rng(seed)
    entity = {node: rng.normal(0, 0.1, size=dim) for node in nodes}
    relation = {rel: rng.normal(0, 0.1, size=dim) for rel in relations}
    node_list = list(nodes)
    for epoch in range(epochs):
        order = rng.permutation(len(triples))
        for index in order:
            head, rel, tail = triples[int(index)]
            corrupt_tail = tail
            while corrupt_tail == tail:
                corrupt_tail = node_list[int(rng.integers(0, len(node_list)))]
            pos_vec = entity[head] + relation[rel] - entity[tail]
            neg_vec = entity[head] + relation[rel] - entity[corrupt_tail]
            pos_score = np.linalg.norm(pos_vec) + 1e-9
            neg_score = np.linalg.norm(neg_vec) + 1e-9
            if margin + pos_score - neg_score > 0:
                pos_grad = pos_vec / pos_score
                neg_grad = neg_vec / neg_score
                entity[head] = entity[head] - lr * (pos_grad - neg_grad)
                relation[rel] = relation[rel] - lr * (pos_grad - neg_grad)
                entity[tail] = entity[tail] + lr * pos_grad
                entity[corrupt_tail] = entity[corrupt_tail] - lr * neg_grad
        for node in entity:
            norm = np.linalg.norm(entity[node])
            if norm > 1.0:
                entity[node] = entity[node] / norm
    return entity, relation


def mean_reciprocal_rank(triples, entity, relation, nodes):
    ranks = []
    node_list = list(nodes)
    for head, rel, tail in triples[: min(80, len(triples))]:
        ranking = transe_rank(entity, relation, head, rel, node_list)
        ordered = [candidate for candidate, score in ranking]
        ranks.append(1.0 / (ordered.index(tail) + 1))
    return float(np.mean(ranks))


ladder = build_graph_ladder()
transe_results = []
for index, rung in enumerate(ladder):
    triples = graph_to_kg(rung)
    relations = sorted(set(rel for head, rel, tail in triples))
    entity, relation = train_transe(triples, list(rung["graph"].nodes()), relations, seed=SEED + index)
    mrr = mean_reciprocal_rank(triples, entity, relation, list(rung["graph"].nodes()))
    rung["triples"] = triples
    rung["entity"] = entity
    rung["relation"] = relation
    rung["mrr"] = mrr
    transe_results.append(mrr)
    print(f"{rung['name']}: triples={len(triples)} relations={len(relations)} MRR={mrr:.3f}")


## Results visualization

Top row: graph/KG panels colored by community labels. Bottom left: TransE mean reciprocal rank across D1-D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for index, rung in enumerate(ladder):
    graph = rung["graph"]
    positions = rung["pos"]
    labels = rung["labels"]
    colors = [labels[node] for node in graph.nodes()]
    nx.draw_networkx(
        graph,
        pos=positions,
        node_color=colors,
        cmap="tab10",
        with_labels=False,
        node_size=55,
        edge_color="#cccccc",
        ax=axes[0, index],
    )
    axes[0, index].set_title(rung["name"])
    axes[0, index].axis("off")
axes[1, 0].plot(range(1, 6), transe_results, marker="o")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("MRR")
axes[1, 0].set_title("TransE ranking quality")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung: one-to-many translations

One head plus one relation vector cannot land exactly on many distant tails. Filtered ranking is an evaluation fix; richer relation models are a modeling fix.

In [ ]:

entity = {
    "head": np.array([0.0, 0.0]),
    "tail_a": np.array([1.0, 0.0]),
    "tail_b": np.array([0.0, 1.0]),
    "tail_c": np.array([-1.0, 0.0]),
}
relation = {"likes": np.array([1.0, 0.0])}
one_to_many = [
    transe_score(entity, relation, ("head", "likes", "tail_a")),
    transe_score(entity, relation, ("head", "likes", "tail_b")),
    transe_score(entity, relation, ("head", "likes", "tail_c")),
]
filtered_candidates = ["tail_a"]
filtered_rank = transe_rank(entity, relation, "head", "likes", filtered_candidates)
print("one-to-many distances", [round(value, 3) for value in one_to_many])
print("single translation cannot be zero for all distant tails")
print("filtered ranking among known-valid tails", filtered_rank)
print("Fix: use filtered evaluation and consider richer relation models for one-to-many facts.")



## Evaluate it + practice

- Metric: compare the rung's MRR against the no-skill baseline `random tail rank` on D1-D5.
- Sanity check: rerun with the fixed seed and verify D1 reproduces the asserted lesson numbers before trusting larger rungs.
- Ablation: use only easy corruptions or ignore filtered ranking for one-to-many facts; the metric should drop or the failure should become visible.
- Failure signal: a high score with a violated split, symmetry, or relation contract is not a real win.
- Inspect the hardest rung by plotting both the graph artifact and the metric curve rather than reading one scalar alone.

Practice prompts:
1. Change only the D3 noise level and predict how the metric curve should move.


2. Replace the D5 seed with another fixed seed and compare the artifact panel.

3. Add one cheap assertion that would catch the lesson pitfall before the metric is printed.